<sub>**The Annotated ML Handbook** · Part III: Pandas · by Mahir Absar Khan</sub>

# Chapter 11: pandas foundations: Series, DataFrames and selection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahir-606/annotated-ml-handbook/blob/main/notebooks/part-3-pandas/11-pandas-basics.ipynb) &nbsp; [📚 Contents](../../book/README.md) &nbsp; [🌐 Web edition](https://mahir-606.github.io/annotated-ml-handbook/)

*Run the setup cell first, then read and run the cells from top to bottom. Every code cell is followed by a table explaining what each line does.*

In [1]:
# Repository setup (not part of the book's examples): run this cell first.
import os, sys
from pathlib import Path

# Files the examples create (CSV, models, figures) go into ./outputs next to the notebook.
OUT = Path.cwd() if Path.cwd().name == "outputs" else Path.cwd() / "outputs"
OUT.mkdir(exist_ok=True)
os.chdir(OUT)

# insurance.csv is the practice dataset created in Chapter 11 and reused by later chapters.
# If it is not here yet, copy it from the repository's data/ folder (or download it in Colab).
if not Path("insurance.csv").exists():
    import shutil, urllib.request
    for base in OUT.parents:
        if (base / "data" / "insurance.csv").exists():
            shutil.copy(base / "data" / "insurance.csv", "insurance.csv")
            break
    else:
        try:
            urllib.request.urlretrieve("https://raw.githubusercontent.com/Mahir-606/"
                                       "annotated-ml-handbook/main/data/insurance.csv", "insurance.csv")
        except Exception as e:
            print("Could not fetch insurance.csv; run Chapter 11 first.", e)

# The figure style used in the book.
import matplotlib.pyplot as plt
from cycler import cycler
plt.rcParams.update({
    "figure.figsize": (7.2, 4.0), "figure.dpi": 100,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.28, "grid.linewidth": 0.6,
    "axes.prop_cycle": cycler(color=["#1f4fff", "#e0245e", "#0f9b62", "#ff6a00",
                                     "#7b3ff2", "#0091b3", "#d99a00", "#d42a2a"]),
    "axes.titleweight": "bold", "axes.titlesize": 12.5, "axes.titlelocation": "left",
    "font.size": 10.5, "legend.frameon": False, "image.cmap": "viridis",
})

# Plotly: interactive in Jupyter/Colab; tools/run_all.py stores static PNGs so GitHub can show them.
try:
    import plotly.io as pio
    pio.renderers.default = os.environ.get("PLOTLY_RENDERER") or (
        "colab" if "google.colab" in sys.modules else "notebook_connected")
except ImportError:
    pass

# pandas foundations: Series, DataFrames and selection

pandas gives you the **DataFrame**: a table with labelled rows and columns where each column can have its own type. It is how nearly every ML project loads, inspects and prepares data. This book uses **pandas 3**, which changed a few long-standing behaviours. Those changes are flagged where they matter.

## 11.1 Series and DataFrame · *Beginner*

> **📖 Definitions**
>
> **Series**: pandas' one-dimensional labelled array: one column of data plus an index of row labels.
>
> **DataFrame**: pandas' two-dimensional labelled table: columns (each a Series, possibly of different types) sharing one row index.
>
> **Index (pandas)**: the row labels of a Series or DataFrame (by default 0, 1, 2, …), used for lookup and for aligning data.


A **Series** is one labelled column. A **DataFrame** is a collection of Series that share the same row labels (the **index**).

**Building both by hand**

In [2]:
import numpy as np
import pandas as pd

s = pd.Series([4.2, 3.9, 5.1], index=["a", "b", "c"], name="yield")
print(s)
print(s["b"], s.mean())

df = pd.DataFrame({
    "plot": ["P1", "P2", "P3", "P4"],
    "fertilizer": ["A", "B", "A", "B"],
    "rain_mm": [120, 95, 140, 80],
    "yield_t": [4.2, 3.9, 5.1, 3.4],
})
print(df)
print(df.shape, list(df.columns), df.index)

a    4.2
b    3.9
c    5.1
Name: yield, dtype: float64
3.9 4.3999999999999995
  plot fertilizer  rain_mm  yield_t
0   P1          A      120      4.2
1   P2          B       95      3.9
2   P3          A      140      5.1
3   P4          B       80      3.4
(4, 4) ['plot', 'fertilizer', 'rain_mm', 'yield_t'] RangeIndex(start=0, stop=4, step=1)


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>import numpy as np</code></td><td>Import NumPy (the array library) under the short name <code>np</code>.</td></tr>
<tr><td><code>import pandas as pd</code></td><td>The universal alias.</td></tr>
<tr><td><code>s = pd.Series([4.2, 3.9, 5.1], index=[&quot;a&quot;, &quot;b&quot;, &quot;c&quot;], name=&quot;yield&quot;)</code></td><td>Values + row labels + a name.</td></tr>
<tr><td><code>print(s)</code></td><td>Display the value of <code>s</code>.</td></tr>
<tr><td><code>print(s[&quot;b&quot;], s.mean())</code></td><td>Label-based lookup; NumPy-style maths.</td></tr>
<tr><td><code>df = pd.DataFrame({</code></td><td>A dict of column name -&gt; list of values.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&quot;plot&quot;: [&quot;P1&quot;, &quot;P2&quot;, &quot;P3&quot;, &quot;P4&quot;],</code></td><td>Column &quot;plot&quot;: a text label for each field plot.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&quot;fertilizer&quot;: [&quot;A&quot;, &quot;B&quot;, &quot;A&quot;, &quot;B&quot;],</code></td><td>Column &quot;fertilizer&quot;: the treatment.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&quot;rain_mm&quot;: [120, 95, 140, 80],</code></td><td>Column &quot;rain_mm&quot;: integers.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&quot;yield_t&quot;: [4.2, 3.9, 5.1, 3.4],</code></td><td>Column &quot;yield_t&quot;: floats.</td></tr>
<tr><td><code>})</code></td><td>Close the dictionary and the call or tuple opened above.</td></tr>
<tr><td><code>print(df)</code></td><td>Display the value of <code>df</code>.</td></tr>
<tr><td><code>print(df.shape, list(df.columns), df.index)</code></td><td>(rows, cols), column names, the default RangeIndex 0..3.</td></tr>
</table>
</details>

## 11.2 The practice dataset · *Beginner*

The rest of Part III uses a realistic synthetic table of 1,000 insurance customers. Generating it yourself means you can reproduce every output in this part exactly.

**Create and save the practice dataset**

In [3]:
rng = np.random.default_rng(2026)
n = 1000
age = rng.integers(18, 65, n)
sex = rng.choice(["female", "male"], n)
bmi = np.round(rng.normal(30, 6, n).clip(16, 53), 1)
children = rng.choice([0, 1, 2, 3, 4], n, p=[.43, .24, .18, .12, .03])
smoker = rng.choice(["yes", "no"], n, p=[.2, .8])
region = rng.choice(["northeast", "northwest", "southeast", "southwest"], n)
charges = (2500 + 260 * age + 320 * (bmi - 30).clip(0) + 450 * children
           + np.where(smoker == "yes", 23000, 0)
           + rng.gamma(2.0, 1500, n)).round(2)
ins = pd.DataFrame({"age": age, "sex": sex, "bmi": bmi, "children": children,
                    "smoker": smoker, "region": region, "charges": charges})
ins.loc[rng.choice(n, 40, replace=False), "bmi"] = np.nan
ins.to_csv("insurance.csv", index=False)
print("saved", ins.shape)

saved (1000, 7)


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>rng = np.random.default_rng(2026)</code></td><td>Fixed seed: everyone gets identical data.</td></tr>
<tr><td><code>n = 1000</code></td><td>Number of customers.</td></tr>
<tr><td><code>age = rng.integers(18, 65, n)</code></td><td>Ages 18..64.</td></tr>
<tr><td><code>sex = rng.choice([&quot;female&quot;, &quot;male&quot;], n)</code></td><td>Random sex, about 50/50.</td></tr>
<tr><td><code>bmi = np.round(rng.normal(30, 6, n).clip(16, 53), 1)</code></td><td>Body-mass index, clipped to a plausible range.</td></tr>
<tr><td><code>children = rng.choice([0, 1, 2, 3, 4], n, p=[.43, .24, .18, .12, .03])</code></td><td>Number of children with realistic probabilities.</td></tr>
<tr><td><code>smoker = rng.choice([&quot;yes&quot;, &quot;no&quot;], n, p=[.2, .8])</code></td><td>About 20% smokers.</td></tr>
<tr><td><code>region = rng.choice([&quot;northeast&quot;, &quot;northwest&quot;, &quot;southeast&quot;, &quot;southwest&quot;], n)</code></td><td>One of four regions.</td></tr>
<tr><td><code>charges = (2500 + 260 * age + 320 * (bmi - 30).clip(0) + 450 * children</code></td><td>Charges grow with age, with BMI above 30 and with children...</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;+ np.where(smoker == &quot;yes&quot;, 23000, 0)</code></td><td>Smokers cost much more: a strong signal to discover later.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;+ rng.gamma(2.0, 1500, n)).round(2)</code></td><td>Right-skewed noise, like real costs.</td></tr>
<tr><td><code>ins = pd.DataFrame({&quot;age&quot;: age, &quot;sex&quot;: sex, &quot;bmi&quot;: bmi, &quot;children&quot;: children,</code></td><td>Assemble the columns into a DataFrame...</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&quot;smoker&quot;: smoker, &quot;region&quot;: region, &quot;charges&quot;: charges})</code></td><td>...continued.</td></tr>
<tr><td><code>ins.loc[rng.choice(n, 40, replace=False), &quot;bmi&quot;] = np.nan</code></td><td>Inject 40 missing BMI values to clean later.</td></tr>
<tr><td><code>ins.to_csv(&quot;insurance.csv&quot;, index=False)</code></td><td>Save WITHOUT the index column (a very common mistake otherwise).</td></tr>
<tr><td><code>print(&quot;saved&quot;, ins.shape)</code></td><td>(1000, 7)</td></tr>
</table>
</details>

## 11.3 Reading and writing data · *Beginner*

**read_csv and friends**

In [4]:
df = pd.read_csv("insurance.csv")
print(df.head())

   age     sex   bmi  children smoker     region   charges
0   58  female  27.9         0     no  northwest  26115.49
1   26    male  39.4         2     no  northwest  15021.19
2   19    male  34.6         2     no  southwest  15319.15
3   48  female  17.3         2     no  southeast  16844.54
4   35    male  23.8         3     no  southwest  17184.32


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>df = pd.read_csv(&quot;insurance.csv&quot;)</code></td><td>Parse a CSV into a DataFrame, inferring column types.</td></tr>
<tr><td><code>print(df.head())</code></td><td>First 5 rows (<code>df.head(10)</code> for 10). <code>df.tail()</code> shows the last rows.</td></tr>
</table>
</details>

| Task | Read | Write |
|---|---|---|
| CSV | `pd.read_csv("f.csv", sep=",", na_values=["?", "NA"])` | `df.to_csv("f.csv", index=False)` |
| Excel | `pd.read_excel("f.xlsx", sheet_name="Sheet1")` (needs `openpyxl`) | `df.to_excel("f.xlsx", index=False)` |
| Parquet (fast, typed) | `pd.read_parquet("f.parquet")` (needs `pyarrow`) | `df.to_parquet("f.parquet")` |
| JSON | `pd.read_json("f.json")` | `df.to_json("f.json", orient="records")` |
| SQL | `pd.read_sql("SELECT * FROM t", conn)` | `df.to_sql("t", conn)` |

> **💡 Tip** · **Useful `read_csv` arguments**
>
> `usecols=[...]` loads only some columns; `dtype={"id": "str"}` forces types; `parse_dates=["date"]` converts date columns; `nrows=1000` previews a huge file; `encoding="latin-1"` fixes `UnicodeDecodeError`; `sep=";"` handles European CSVs; `index_col=0` uses the first column as the index.

## 11.4 Inspecting a DataFrame · *Beginner*

These five calls are the first thing to run on any new dataset.

**First look**

In [5]:
print(df.shape)
df.info()

(1000, 7)
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1000 non-null   int64  
 1   sex       1000 non-null   str    
 2   bmi       960 non-null    float64
 3   children  1000 non-null   int64  
 4   smoker    1000 non-null   str    
 5   region    1000 non-null   str    
 6   charges   1000 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 54.8 KB


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>print(df.shape)</code></td><td>(1000, 7): rows, columns.</td></tr>
<tr><td><code>df.info()</code></td><td>Column names, non-null counts, dtypes and memory. Reveals missing values at a glance.</td></tr>
</table>
</details>

**Summary statistics**

In [6]:
print(df.describe().round(2))
print(df.describe(include="str"))

           age     bmi  children   charges
count  1000.00  960.00   1000.00   1000.00
mean     40.71   29.52      1.02  21473.52
std      13.50    5.84      1.16   9881.93
min      18.00   16.00      0.00   7375.92
25%      29.00   25.30      0.00  14669.86
50%      40.00   29.50      1.00  18670.80
75%      52.00   33.60      2.00  23315.51
max      64.00   50.00      4.00  50817.04
         sex smoker     region
count   1000   1000       1000
unique     2      2          4
top     male     no  southeast
freq     538    816        266


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>print(df.describe().round(2))</code></td><td>count, mean, std, min, quartiles, max for numeric columns.</td></tr>
<tr><td><code>print(df.describe(include=&quot;str&quot;))</code></td><td>For text columns: count, number of unique values, most frequent (top), its frequency.</td></tr>
</table>
</details>

**Counts and missing values**

In [7]:
print(df["region"].value_counts())
print(df["smoker"].value_counts(normalize=True))
print(df.isna().sum())
print(df.nunique())

region
southeast    266
southwest    257
northeast    247
northwest    230
Name: count, dtype: int64
smoker
no     0.816
yes    0.184
Name: proportion, dtype: float64
age          0
sex          0
bmi         40
children     0
smoker       0
region       0
charges      0
dtype: int64
age           47
sex            2
bmi          244
children       5
smoker         2
region         4
charges     1000
dtype: int64


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>print(df[&quot;region&quot;].value_counts())</code></td><td>Frequency of each category, sorted.</td></tr>
<tr><td><code>print(df[&quot;smoker&quot;].value_counts(normalize=True))</code></td><td>As proportions: check class balance before modelling!</td></tr>
<tr><td><code>print(df.isna().sum())</code></td><td>Missing values per column.</td></tr>
<tr><td><code>print(df.nunique())</code></td><td>Number of distinct values per column.</td></tr>
</table>
</details>

> **📝 Note** · **What changed in pandas 3**
>
> Text columns now get a dedicated string dtype, shown as `str`, instead of the old catch-all `object`. That is why the call above is `describe(include="str")`. And **Copy-on-Write** is always on: any DataFrame you derive from another behaves like an independent copy. See section 11.7.

## 11.5 Selecting data · *Beginner*

> **📖 Definitions**
>
> **`.loc` and `.iloc`**: `.loc[rows, cols]` selects by labels (slices include the end); `.iloc[rows, cols]` selects by integer positions (slices exclude the end).


### 11.5.1 Columns

**One column, several columns**

In [8]:
ages = df["age"]
print(type(ages).__name__, ages.shape)
sub = df[["age", "bmi", "charges"]]
print(type(sub).__name__, sub.shape)
print(df.age.mean())

Series (1000,)
DataFrame (1000, 3)
40.707


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>ages = df[&quot;age&quot;]</code></td><td>Single brackets + one name -&gt; a Series.</td></tr>
<tr><td><code>print(type(ages).__name__, ages.shape)</code></td><td>Series, shape (1000,).</td></tr>
<tr><td><code>sub = df[[&quot;age&quot;, &quot;bmi&quot;, &quot;charges&quot;]]</code></td><td>Double brackets (a LIST of names) -&gt; a DataFrame.</td></tr>
<tr><td><code>print(type(sub).__name__, sub.shape)</code></td><td>DataFrame, shape (1000, 3).</td></tr>
<tr><td><code>print(df.age.mean())</code></td><td>Attribute access works for simple names, but df[&quot;age&quot;] always works. Prefer it.</td></tr>
</table>
</details>

### 11.5.2 Rows: `.loc` (labels) and `.iloc` (positions)

`.loc[rows, cols]` selects by **label**; `.iloc[rows, cols]` selects by **integer position**, like NumPy.

**loc vs iloc**

In [9]:
print(df.loc[0, "charges"])
print(df.loc[0:2, ["age", "smoker"]])
print(df.iloc[0:2, 0:3])
print(df.iloc[-1])

26115.49
   age smoker
0   58     no
1   26     no
2   19     no
   age     sex   bmi
0   58  female  27.9
1   26    male  39.4
age                25
sex            female
bmi              27.6
children            0
smoker            yes
region      northwest
charges      33475.22
Name: 999, dtype: object


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>print(df.loc[0, &quot;charges&quot;])</code></td><td>Row label 0, column &quot;charges&quot;.</td></tr>
<tr><td><code>print(df.loc[0:2, [&quot;age&quot;, &quot;smoker&quot;]])</code></td><td>.loc slices INCLUDE the end label: rows 0, 1 AND 2.</td></tr>
<tr><td><code>print(df.iloc[0:2, 0:3])</code></td><td>.iloc slices EXCLUDE the end, like Python: rows 0-1, cols 0-2.</td></tr>
<tr><td><code>print(df.iloc[-1])</code></td><td>Last row, as a Series.</td></tr>
</table>
</details>

> **⚠️ Pitfall** · **`.loc` includes the endpoint, `.iloc` does not**
>
> `df.loc[0:2]` returns 3 rows; `df.iloc[0:2]` returns 2. After filtering or sorting, row labels no longer match positions. Use `.iloc` when you mean "the first n rows".

### 11.5.3 Filtering with conditions

**Boolean filtering**

In [10]:
smokers = df[df["smoker"] == "yes"]
print(len(smokers))
older_smokers = df[(df["age"] >= 50) & (df["smoker"] == "yes")]
print(len(older_smokers))
south = df[df["region"].isin(["southeast", "southwest"])]
print(south["region"].unique())
print(df.loc[df["bmi"] > 45, ["age", "bmi"]].head(3))
print(df.query("age < 25 and children >= 2").shape)
limit = 40
print(df.query("bmi > @limit").shape)

184
61
<StringArray>
['southwest', 'southeast']
Length: 2, dtype: str
     age   bmi
237   28  45.1
304   20  50.0
621   26  48.4
(40, 7)
(33, 7)


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>smokers = df[df[&quot;smoker&quot;] == &quot;yes&quot;]</code></td><td>Keep rows where the condition is True.</td></tr>
<tr><td><code>print(len(smokers))</code></td><td>Number of smokers.</td></tr>
<tr><td><code>older_smokers = df[(df[&quot;age&quot;] &gt;= 50) &amp; (df[&quot;smoker&quot;] == &quot;yes&quot;)]</code></td><td>Combine with &amp; | ~ and PARENTHESES, as in NumPy.</td></tr>
<tr><td><code>print(len(older_smokers))</code></td><td>Smokers aged 50 or more.</td></tr>
<tr><td><code>south = df[df[&quot;region&quot;].isin([&quot;southeast&quot;, &quot;southwest&quot;])]</code></td><td>Membership in a list of values.</td></tr>
<tr><td><code>print(south[&quot;region&quot;].unique())</code></td><td>Only the two southern regions remain.</td></tr>
<tr><td><code>print(df.loc[df[&quot;bmi&quot;] &gt; 45, [&quot;age&quot;, &quot;bmi&quot;]].head(3))</code></td><td>Filter rows AND choose columns in one .loc.</td></tr>
<tr><td><code>print(df.query(&quot;age &lt; 25 and children &gt;= 2&quot;).shape)</code></td><td><code>query</code> takes a readable string expression.</td></tr>
<tr><td><code>limit = 40</code></td><td>A Python variable to use inside query().</td></tr>
<tr><td><code>print(df.query(&quot;bmi &gt; @limit&quot;).shape)</code></td><td><code>@name</code> refers to a Python variable inside query.</td></tr>
</table>
</details>

## 11.6 Adding, changing and removing columns · *Beginner*

**Creating new columns**

In [11]:
df = pd.read_csv("insurance.csv")
df["charges_k"] = df["charges"] / 1000
df["is_smoker"] = (df["smoker"] == "yes").astype(int)
df["age_group"] = pd.cut(df["age"], bins=[17, 30, 45, 64],
                         labels=["18-30", "31-45", "46-64"])
df = df.assign(log_charges=np.log(df["charges"]))
df.loc[df["children"] >= 3, "big_family"] = True
df = df.rename(columns={"bmi": "BMI"})
df = df.drop(columns=["big_family"])
print(df.head(3))

   age     sex   BMI  children smoker     region   charges  charges_k  \
0   58  female  27.9         0     no  northwest  26115.49   26.11549   
1   26    male  39.4         2     no  northwest  15021.19   15.02119   
2   19    male  34.6         2     no  southwest  15319.15   15.31915   

   is_smoker age_group  log_charges  
0          0     46-64    10.170284  
1          0     18-30     9.617217  
2          0     18-30     9.636859  


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>df = pd.read_csv(&quot;insurance.csv&quot;)</code></td><td>Reload the original data.</td></tr>
<tr><td><code>df[&quot;charges_k&quot;] = df[&quot;charges&quot;] / 1000</code></td><td>Vectorized arithmetic creates a new column.</td></tr>
<tr><td><code>df[&quot;is_smoker&quot;] = (df[&quot;smoker&quot;] == &quot;yes&quot;).astype(int)</code></td><td>Boolean -&gt; 0/1: a numeric feature.</td></tr>
<tr><td><code>df[&quot;age_group&quot;] = pd.cut(df[&quot;age&quot;], bins=[17, 30, 45, 64],</code></td><td>Bin a numeric column into labelled intervals.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;labels=[&quot;18-30&quot;, &quot;31-45&quot;, &quot;46-64&quot;])</code></td><td>Names for the three bins.</td></tr>
<tr><td><code>df = df.assign(log_charges=np.log(df[&quot;charges&quot;]))</code></td><td><code>assign</code> returns a new DataFrame: good for chaining.</td></tr>
<tr><td><code>df.loc[df[&quot;children&quot;] &gt;= 3, &quot;big_family&quot;] = True</code></td><td>Conditional assignment; other rows get NaN.</td></tr>
<tr><td><code>df = df.rename(columns={&quot;bmi&quot;: &quot;BMI&quot;})</code></td><td>Rename columns.</td></tr>
<tr><td><code>df = df.drop(columns=[&quot;big_family&quot;])</code></td><td>Remove a column (axis=1). Rows: df.drop(index=[...]).</td></tr>
<tr><td><code>print(df.head(3))</code></td><td>The first three rows with the new columns.</td></tr>
</table>
</details>

### 11.6.1 Sorting

**sort_values and nlargest**

In [12]:
print(df.sort_values("charges", ascending=False).head(3)[["age", "smoker", "charges"]])
print(df.sort_values(["smoker", "age"], ascending=[True, False]).head(3)[["smoker", "age"]])
print(df.nlargest(3, "BMI")[["BMI", "age"]])

     age smoker   charges
274   62    yes  50817.04
556   62    yes  50760.11
918   59    yes  48079.72
    smoker  age
50      no   64
131     no   64
160     no   64
      BMI  age
304  50.0   20
621  48.4   26
698  46.9   47


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>print(df.sort_values(&quot;charges&quot;, ascending=False).head(3)[[&quot;age&quot;, &quot;smoker&quot;, &quot;charges&quot;]])</code></td><td>Most expensive first.</td></tr>
<tr><td><code>print(df.sort_values([&quot;smoker&quot;, &quot;age&quot;], ascending=[True, False]).head(3)[[&quot;smoker&quot;, &quot;age&quot;]])</code></td><td>Sort by several keys.</td></tr>
<tr><td><code>print(df.nlargest(3, &quot;BMI&quot;)[[&quot;BMI&quot;, &quot;age&quot;]])</code></td><td>Shortcut for the top-n rows by a column.</td></tr>
</table>
</details>

## 11.7 Copy-on-Write and the "chained assignment" trap · *Intermediate*

> **📖 Definitions**
>
> **Copy-on-Write**: pandas 3 behaviour in which any DataFrame derived from another acts as an independent copy; the data is only copied when one of them is modified.
>
> **Chained assignment**: two indexing operations followed by `=`, as in `df[mask]["col"] = v`. In pandas 3 it never modifies the original; use one `.loc` call instead.


In pandas 3, every new DataFrame derived from another (by filtering, selecting or slicing) behaves like an independent copy. One consequence: **chained assignment**, meaning two indexing operations in a row followed by `=`, never modifies the original.

**This does NOT change df**

In [13]:
df = pd.read_csv("insurance.csv")
df[df["age"] > 60]["bmi"] = 0
print((df["bmi"] == 0).sum())

0


/tmp/ipykernel_422/2665612345.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  df[df["age"] > 60]["bmi"] = 0


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>df = pd.read_csv(&quot;insurance.csv&quot;)</code></td><td>Reload the original data.</td></tr>
<tr><td><code>df[df[&quot;age&quot;] &gt; 60][&quot;bmi&quot;] = 0</code></td><td>Chained: df[...] makes a new object, and [&quot;bmi&quot;] = 0 modifies only that temporary.</td></tr>
<tr><td><code>print((df[&quot;bmi&quot;] == 0).sum())</code></td><td>Still 0 rows changed: the original df is untouched.</td></tr>
</table>
</details>

**The correct way: one .loc call**

In [14]:
df.loc[df["age"] > 60, "bmi"] = 0
print((df["bmi"] == 0).sum())
df = pd.read_csv("insurance.csv")
young = df[df["age"] < 30].copy()
young["bmi"] = young["bmi"].round()

82


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>df.loc[df[&quot;age&quot;] &gt; 60, &quot;bmi&quot;] = 0</code></td><td>Row condition and column in ONE indexing call modifies df itself.</td></tr>
<tr><td><code>print((df[&quot;bmi&quot;] == 0).sum())</code></td><td>Count rows whose BMI is now 0.</td></tr>
<tr><td><code>df = pd.read_csv(&quot;insurance.csv&quot;)</code></td><td>Reload the clean data.</td></tr>
<tr><td><code>young = df[df[&quot;age&quot;] &lt; 30].copy()</code></td><td>Want a separate subset to edit? Take it explicitly (the .copy() makes intent clear).</td></tr>
<tr><td><code>young[&quot;bmi&quot;] = young[&quot;bmi&quot;].round()</code></td><td>Editing the subset never affects df.</td></tr>
</table>
</details>

> **🐞 Bug hunt** · **`SettingWithCopyWarning` / `ChainedAssignmentError`**
>
> Older pandas printed `SettingWithCopyWarning`; pandas 3 warns about chained assignment instead, and the assignment has no effect on the original. The fix is always the same: use a single `df.loc[row_condition, "column"] = value`.

## 11.8 The index · *Intermediate*

**Setting and resetting the index**

In [15]:
idx = df.set_index("region")
print(idx.loc["southwest"].shape)
print(idx.reset_index().columns.tolist()[:3])
r = df[df["age"] > 60]
print(r.index[:5].tolist())
print(r.reset_index(drop=True).index[:5].tolist())

(257, 6)
['region', 'age', 'sex']
[21, 23, 50, 60, 62]
[0, 1, 2, 3, 4]


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>idx = df.set_index(&quot;region&quot;)</code></td><td>Move a column into the row labels.</td></tr>
<tr><td><code>print(idx.loc[&quot;southwest&quot;].shape)</code></td><td>Fast selection of all rows with that label.</td></tr>
<tr><td><code>print(idx.reset_index().columns.tolist()[:3])</code></td><td><code>reset_index</code> moves the index back into a column.</td></tr>
<tr><td><code>r = df[df[&quot;age&quot;] &gt; 60]</code></td><td>Customers over 60.</td></tr>
<tr><td><code>print(r.index[:5].tolist())</code></td><td>Filtering keeps the ORIGINAL labels...</td></tr>
<tr><td><code>print(r.reset_index(drop=True).index[:5].tolist())</code></td><td>...reset_index(drop=True) renumbers 0..n-1 and discards the old labels.</td></tr>
</table>
</details>

> **🐞 Bug hunt** · **Rows misalign after combining a filtered DataFrame with something else**
>
> pandas aligns operations by **index label**, not by position. If you filtered or sorted a DataFrame and then assign a NumPy array or another Series, labels may not line up and you get `NaN`s or shuffled values. Use `.reset_index(drop=True)` on both, or assign `.to_numpy()` when you mean "by position".

## 11.9 From pandas to NumPy and back · *Beginner*

**Getting arrays out for modelling**

In [16]:
X = df[["age", "bmi", "children"]].to_numpy()
y = df["charges"].to_numpy()
print(X.shape, X.dtype, y[:3])
back = pd.DataFrame(X[:2], columns=["age", "bmi", "children"])
print(back)

(1000, 3) float64 [26115.49 15021.19 15319.15]
    age   bmi  children
0  58.0  27.9       0.0
1  26.0  39.4       2.0


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>X = df[[&quot;age&quot;, &quot;bmi&quot;, &quot;children&quot;]].to_numpy()</code></td><td>A 2-D NumPy array (n_samples, n_features).</td></tr>
<tr><td><code>y = df[&quot;charges&quot;].to_numpy()</code></td><td>A 1-D target array.</td></tr>
<tr><td><code>print(X.shape, X.dtype, y[:3])</code></td><td>Feature matrix shape and type, and the first three targets.</td></tr>
<tr><td><code>back = pd.DataFrame(X[:2], columns=[&quot;age&quot;, &quot;bmi&quot;, &quot;children&quot;])</code></td><td>Wrap an array back into a labelled table.</td></tr>
<tr><td><code>print(back)</code></td><td>Display the value of <code>back</code>.</td></tr>
</table>
</details>

scikit-learn accepts DataFrames directly and remembers the column names, so in practice you often skip `to_numpy()` entirely.

> **✍️ Try it yourself**
>
> 1. How many female smokers in the southeast are over 40?
> 2. Add a column `bmi_class` with "under" (< 18.5), "normal" (< 25), "over" (< 30), "obese" (≥ 30) using `pd.cut`.
> 3. Show the 5 youngest customers with 4 children, sorted by charges.
>
> *Worked solutions:* [solutions-parts-1-4.ipynb](../../solutions/solutions-parts-1-4.ipynb), section *Chapter 11*.

---

⬅️ [Chapter 10: Machine learning from scratch with NumPy](../part-2-numpy/10-numpy-ml-scratch.ipynb) · [📚 Contents](../../book/README.md) · [Chapter 12: Cleaning data with pandas](12-pandas-cleaning.ipynb) ➡️